# Day 034 Project Solution — Installable AI CLI Tool

A multi-command `AICli` wrapping argparse + ollama. Demonstrates `make_parser`, `make_subcommand_parser`, `dispatch`, and `AICli`.

In [ ]:
from argparse import ArgumentParser

def make_parser() -> ArgumentParser:
    parser = ArgumentParser(
        prog='ai-tool',
        description='AI command-line tool powered by local LLM',
    )
    parser.add_argument(
        '--prompt', '-p', type=str, required=True,
        help='Prompt to send to the model',
    )
    parser.add_argument(
        '--model', '-m', type=str, default='llama3.2',
        help='Ollama model name (default: llama3.2)',
    )
    parser.add_argument(
        '--verbose', '-v', action='store_true',
        help='Print extra diagnostic output',
    )
    return parser


def make_extended_parser() -> ArgumentParser:
    parser = ArgumentParser(
        prog='ai-batch',
        description='AI batch processing tool',
    )
    parser.add_argument(
        '--prompt', '-p', type=str, required=True,
        help='Prompt text',
    )
    parser.add_argument(
        '--count', '-n', type=int, default=1,
        help='Number of completions (default: 1)',
    )
    parser.add_argument(
        '--format', '-f',
        choices=['text', 'json', 'markdown'],
        default='text',
        help='Output format (default: text)',
    )
    parser.add_argument(
        '--temperature', type=float, default=0.7,
        help='Sampling temperature 0.0-1.0 (default: 0.7)',
    )
    return parser


def make_subcommand_parser() -> ArgumentParser:
    parser = ArgumentParser(prog='ai-tool', description='AI CLI')
    subs   = parser.add_subparsers(dest='command', required=True,
                                   title='commands')

    chat = subs.add_parser('chat', help='Send a prompt to the AI')
    chat.add_argument('--prompt', '-p', required=True, help='The prompt')
    chat.add_argument('--model',  '-m', default='llama3.2')

    summarize = subs.add_parser('summarize', help='Summarize text')
    summarize.add_argument('--text',  '-t', required=True,
                           help='Text to summarize')
    summarize.add_argument('--model', '-m', default='llama3.2')

    return parser


def dispatch(ns, handlers: dict) -> str:
    cmd = ns.command
    if cmd not in handlers:
        raise KeyError(f"No handler registered for command: {cmd!r}")
    return handlers[cmd](ns)


import ollama
from argparse import ArgumentParser

class AICli:
    def __init__(self, prog: str = 'ai-tool', model: str = 'llama3.2'):
        self.model       = model
        self._parser     = ArgumentParser(prog=prog,
                                          description='AI command-line tool')
        self._subs       = self._parser.add_subparsers(dest='command',
                                                        required=True)
        self._prompt_fns: dict = {}

    def add_command(self, name: str, prompt_fn,
                    help: str = '') -> 'AICli':
        sub = self._subs.add_parser(name, help=help)
        sub.add_argument('--input', '-i', required=True, help='Input text')
        self._prompt_fns[name] = prompt_fn
        return self

    def run(self, args_list: list[str]) -> str:
        ns     = self._parser.parse_args(args_list)
        prompt = self._prompt_fns[ns.command](ns.input)
        resp   = ollama.chat(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return resp['message']['content']

## Action 1 — Demonstrate make_parser and make_extended_parser

In [ ]:
import io, sys

# make_parser
p = make_parser()
ns = p.parse_args(['--prompt', 'hello world', '--verbose'])
assert ns.prompt  == 'hello world'
assert ns.model   == 'llama3.2'
assert ns.verbose == True
print(f'make_parser: prompt={ns.prompt!r}, model={ns.model!r}, verbose={ns.verbose}')

# make_extended_parser
ep = make_extended_parser()
ns2 = ep.parse_args(['--prompt', 'hi', '--count', '3', '--format', 'json'])
assert ns2.count == 3               # int
assert ns2.format == 'json'          # choice
assert isinstance(ns2.temperature, float)
print(f'make_extended_parser: count={ns2.count} (int), format={ns2.format!r}, '
      f'temp={ns2.temperature}')

## Action 2 — Subcommands and dispatch

In [ ]:
def _chat_handler(ns):
    return f'[chat] prompt={ns.prompt!r}, model={ns.model!r}'

def _summarize_handler(ns):
    return f'[summarize] text={ns.text[:30]!r}, model={ns.model!r}'

handlers = {'chat': _chat_handler, 'summarize': _summarize_handler}
sp = make_subcommand_parser()

ns = sp.parse_args(['chat', '--prompt', 'What is AI?'])
print(dispatch(ns, handlers))

ns2 = sp.parse_args(['summarize', '--text', 'Long article about AI...'])
print(dispatch(ns2, handlers))

assert ns.command  == 'chat'
assert ns2.command == 'summarize'

## Action 3 — AICli end-to-end

In [ ]:
cli = (
    AICli(prog='ai-tool', model='llama3.2')
    .add_command('ask',
                 lambda t: t,
                 help='Send raw prompt')
    .add_command('sentiment',
                 lambda t: f"Classify as positive/negative/neutral. One word.\n\n'{t}'",
                 help='Classify sentiment')
)

r1 = cli.run(['ask', '--input', 'What colour is the sky? One word.'])
print(f'ask:       {r1.strip()!r}')

r2 = cli.run(['sentiment', '--input', 'I love this product!'])
print(f'sentiment: {r2.strip()!r}')

assert isinstance(r1, str) and r1.strip()
assert isinstance(r2, str) and r2.strip()

# Entry-point reminder
print('\n# pyproject.toml entry point:')
print('[project.scripts]')
print('ai-tool = "my_module.cli:main"')

print('\nCLI complete!')